# 01 — `midas_zipper`: raw frames → `*.MIDAS.zip`

`midas_zipper` turns raw detector data (HDF5 / GE / TIFF / CBF) into
the `*.MIDAS.zip` Zarr archive the rest of MIDAS consumes. It is
pip-portable: pure `numpy` / `h5py` / `zarr` / `numcodecs` / `numba`
/ `pillow` — no MIDAS source tree, no C binaries.

This notebook is fully **self-contained**: we synthesize a small
multi-frame TIFF stack + a minimal parameter file, call
`generate_ff_zip()`, inspect the Zarr keys inside the resulting
`.MIDAS.zip`, and finish by patching a metadata key with
`midas-update-zarr`. Runs in a few seconds on CPU.


In [1]:
import tempfile
from pathlib import Path

import numpy as np
import tifffile
import zarr

import midas_zipper
from midas_zipper import generate_ff_zip

print("midas_zipper version:", midas_zipper.__version__)

WORK = Path(tempfile.mkdtemp(prefix="midas_zipper_nb_"))
RAW = WORK / "raw"
OUT = WORK / "out"
RAW.mkdir(); OUT.mkdir()
print("workspace:", WORK)


midas_zipper version: 0.1.1
workspace: /var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/midas_zipper_nb_ghb6k141


## 1. Synthesize a raw TIFF frame stack

A real FF scan is one frame per ω step. We make a tiny **8-frame**
stack of 64×64 `uint16` images, each with a couple of bright Gaussian
"diffraction spots" on a low background. The filename follows the
MIDAS convention `<FileStem>_<zero-padded-number><Ext>` so the zipper
can locate the sequence.


In [2]:
NR_PIXELS = 64
N_FRAMES = 8
rng = np.random.default_rng(0)

def make_frame(i):
    img = rng.integers(80, 120, size=(NR_PIXELS, NR_PIXELS)).astype(np.uint16)
    yy, xx = np.mgrid[0:NR_PIXELS, 0:NR_PIXELS]
    # Two spots that drift with frame index (stand-ins for Bragg peaks).
    for (cy, cx, amp) in [(20 + i, 40, 1500), (45, 15 + i, 1200)]:
        img += (amp * np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / 8.0)).astype(np.uint16)
    return img

FILE_STEM = "synthFF"
EXT = ".tif"
START_NR = 1
for i in range(N_FRAMES):
    fn = RAW / f"{FILE_STEM}_{START_NR + i:06d}{EXT}"
    tifffile.imwrite(fn, make_frame(i))

frames = sorted(RAW.glob(f"{FILE_STEM}_*{EXT}"))
print(f"wrote {len(frames)} frames, e.g. {frames[0].name}")
print("frame shape/dtype:", tifffile.imread(frames[0]).shape, tifffile.imread(frames[0]).dtype)


wrote 8 frames, e.g. synthFF_000001.tif
frame shape/dtype: (64, 64) uint16


## 2. Minimal parameter file

`generate_ff_zip` reads a MIDAS-style param file. For the TIFF path
it needs the file-layout keys (`RawFolder`, `FileStem`, `Ext`,
`Padding`, `StartFileNrFirstLayer`) plus the geometry / scan metadata
that gets copied into the Zarr's `analysis_parameters` group. We keep
it minimal but representative.


In [3]:
PARAM = f'''\
RawFolder {RAW}
FileStem {FILE_STEM}
Ext {EXT}
Padding 6
StartFileNrFirstLayer {START_NR}
NrFilesPerSweep 1
LayerNr 1
OmegaStart 0
OmegaStep 0.25
Wavelength 0.172979
Lsd 1000000.0
px 200.0
BC 32 32
NrPixels {NR_PIXELS}
SpaceGroup 225
LatticeConstant 4.08 4.08 4.08 90 90 90
RingThresh 1 100
RingThresh 2 100
'''
param_fn = WORK / "Parameters.txt"
param_fn.write_text(PARAM)
print(param_fn.read_text())


RawFolder /var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/midas_zipper_nb_ghb6k141/raw
FileStem synthFF
Ext .tif
Padding 6
StartFileNrFirstLayer 1
NrFilesPerSweep 1
LayerNr 1
OmegaStart 0
OmegaStep 0.25
Wavelength 0.172979
Lsd 1000000.0
px 200.0
BC 32 32
NrPixels 64
SpaceGroup 225
LatticeConstant 4.08 4.08 4.08 90 90 90
RingThresh 1 100
RingThresh 2 100



## 3. Generate the `*.MIDAS.zip`

`generate_ff_zip` is a thin programmatic wrapper around the
`midas-ff-zip` CLI — it populates `sys.argv` and runs the same
`main()`. It reads each TIFF, stacks the frames into the Zarr
`exchange/data` array, and writes the parameters into the metadata
groups. Returns 0 on success.


In [4]:
rc = generate_ff_zip(
    result_folder=str(OUT),
    param_file=str(param_fn),
    layer_nr=1,
    num_files_per_scan=N_FRAMES,   # one TIFF == one ω frame; stack all 8
)
print("generate_ff_zip return code:", rc)

zips = list(OUT.glob("*.MIDAS.zip"))
assert zips, "no .MIDAS.zip produced"
zip_fn = zips[0]
print("produced:", zip_fn.name, f"({zip_fn.stat().st_size} bytes)")


Info: 'SkipFrame' not found in parameter file. Defaulting to 0.
Input Data: /var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/midas_zipper_nb_ghb6k141/raw/synthFF_000001.tif
Parameter File: /var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/midas_zipper_nb_ghb6k141/Parameters.txt
Output Zarr: /var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/midas_zipper_nb_ghb6k141/out/synthFF_000001.MIDAS.zip

--- Processing TIFF File(s) ---
Handling as uint32. Files per scan: 8
  - Inferred dimensions from TIFF: 64 x 64
Scan: 8 file(s) found, 1 frames/file. Skipping 0 from files 2+. Total frames to write: 8.
  - Writing actual dark and bright frame data.


  - Wrote frame 1/8 to Zarr archive.
  - Wrote frame 2/8 to Zarr archive.
  - Wrote frame 3/8 to Zarr archive.
  - Wrote frame 4/8 to Zarr archive.
  - Wrote frame 5/8 to Zarr archive.
  - Wrote frame 6/8 to Zarr archive.
  - Wrote frame 7/8 to Zarr archive.
  - Wrote frame 8/8 to Zarr archive.

Written datatype for C-code: 'uint32'

Writing analysis parameters to Zarr file...
RingThresh RingThresh [[  1 100]
 [  2 100]] 2
  - Info: Parameter 'RingsToExclude' not found. Creating with default values.
  - Info: Parameter 'OmegaRanges' not found. Creating with default values.
  - Info: Parameter 'BoxSizes' not found. Creating with default values.
  - Info: Parameter 'ImTransOpt' not found. Creating with default values.

--- Zarr File Structure Verification ---
/
 ├── analysis
 │   └── process
 │       └── analysis_parameters
 │           ├── BoxSizes (1, 4) float64
 │           ├── Ext (1,) |S4
 │           ├── FileStem (1,) |S7
 │           ├── ImTransOpt (1,) int32
 │           ├── Latt

Dataset: /analysis/process/analysis_parameters/Ext
  - Shape:   (1,)
  - Chunks:  (1,)
  - Dtype:   |S4
  - Compressor: {'id': 'blosc', 'cname': 'lz4', 'clevel': 5, 'shuffle': 1, 'blocksize': 0}
-------------------------
Dataset: /analysis/process/analysis_parameters/FileStem
  - Shape:   (1,)
  - Chunks:  (1,)
  - Dtype:   |S7
  - Compressor: {'id': 'blosc', 'cname': 'lz4', 'clevel': 5, 'shuffle': 1, 'blocksize': 0}
-------------------------
Dataset: /analysis/process/analysis_parameters/ImTransOpt
  - Shape:   (1,)
  - Chunks:  (1,)
  - Dtype:   int32
  - Compressor: {'id': 'blosc', 'cname': 'lz4', 'clevel': 5, 'shuffle': 1, 'blocksize': 0}
-------------------------
Dataset: /analysis/process/analysis_parameters/LatticeParameter
  - Shape:   (6,)
  - Chunks:  (6,)
  - Dtype:   float64
  - Compressor: {'id': 'blosc', 'cname': 'lz4', 'clevel': 5, 'shuffle': 1, 'blocksize': 0}
-------------------------
Dataset: /analysis/process/analysis_parameters/LayerNr
  - Shape:   (1,)
  - Chunks: 

## 4. Inspect the Zarr archive

The `.MIDAS.zip` is a zipped Zarr store. We open it read-only and
walk its hierarchy: the frame data lives under `exchange/data`, and
the parameters we passed live under
`analysis/process/analysis_parameters`.


In [5]:
zf = zarr.open(str(zip_fn), mode="r")

print("Top-level groups/arrays:")
def walk(g, prefix=""):
    for name, arr in g.arrays():
        print(f"  [array] {prefix}{name:20s} shape={arr.shape} dtype={arr.dtype}")
    for name, sub in g.groups():
        print(f"  [group] {prefix}{name}/")
        walk(sub, prefix=f"{prefix}{name}/")
walk(zf)

data = zf["exchange/data"]
print()
print("exchange/data shape:", data.shape, " dtype:", data.dtype)
print("frame 0 max / mean :", int(data[0].max()), float(data[0].mean()))


Top-level groups/arrays:
  [group] analysis/
  [group] analysis/process/
  [group] analysis/process/analysis_parameters/
  [array] analysis/process/analysis_parameters/BoxSizes             shape=(1, 4) dtype=float64
  [array] analysis/process/analysis_parameters/Ext                  shape=(1,) dtype=|S4
  [array] analysis/process/analysis_parameters/FileStem             shape=(1,) dtype=|S7
  [array] analysis/process/analysis_parameters/ImTransOpt           shape=(1,) dtype=int32
  [array] analysis/process/analysis_parameters/LatticeParameter     shape=(6,) dtype=float64
  [array] analysis/process/analysis_parameters/LayerNr              shape=(1,) dtype=int32
  [array] analysis/process/analysis_parameters/Lsd                  shape=(1,) dtype=float64
  [array] analysis/process/analysis_parameters/NrFilesPerSweep      shape=(1,) dtype=int32
  [array] analysis/process/analysis_parameters/NrPixels             shape=(1,) dtype=int64
  [array] analysis/process/analysis_parameters/OmegaRang

In [6]:
# A few of the parameters we wrote, read back out of the metadata group.
ap = "analysis/process/analysis_parameters"
for key in ("Wavelength", "Lsd", "YCen", "ZCen", "RingThresh"):
    full = f"{ap}/{key}"
    try:
        print(f"  {key:12s} = {zf[full][...]}")
    except KeyError:
        print(f"  {key:12s} = <not present>")


  Wavelength   = [0.172979]
  Lsd          = [1000000.]
  YCen         = [32.]
  ZCen         = [32.]
  RingThresh   = [[  1. 100.]
 [  2. 100.]]


## 5. Patch a metadata key with `midas-update-zarr`

After a zip exists you often need to tweak one scalar (a refined
beam-center, a tilt) without regenerating the whole archive.
`midas-update-zarr` does an in-place key update. We invoke its
`main()` directly with a populated `argv` (same as the CLI). It
`chdir`s into `-folder` and rewrites the key inside the existing
`.zip` with the system `zip` tool, so we pass the archive's basename
plus its folder. Here we bump `YCen` to a refined value.


In [7]:
import sys
from midas_zipper import update_zarr

key = "analysis/process/analysis_parameters/YCen"
print("YCen before:", zf[key][...])

saved = sys.argv
try:
    sys.argv = [
        "midas-update-zarr",
        "-fn", zip_fn.name,        # basename; update_zarr chdir's to -folder
        "-folder", str(OUT),
        "-keyToUpdate", key,
        "-updatedValue", "33.5",
    ]
    update_zarr.main()
finally:
    sys.argv = saved

# Re-open to see the patched value.
zf2 = zarr.open(str(zip_fn), mode="r")
print("YCen after :", zf2[key][...])


YCen before: [32.]
Initial value: [32.]
updating: analysis/process/analysis_parameters/YCen/.zarray (deflated 47%)
updating: analysis/process/analysis_parameters/YCen/0 (deflated 25%)
Updated value: analysis/process/analysis_parameters/YCen:[33.5]
(1,)
YCen after : [33.5]


## Recap

| Step | Call |
|---|---|
| raw frames → archive | `generate_ff_zip(result_folder=, param_file=, layer_nr=)` |
| inspect | `zarr.open(zip, "r")` → walk `exchange/data` + `analysis/process/analysis_parameters` |
| patch one key in place | `midas-update-zarr -fn … -keyToUpdate … -updatedValue …` |

The same generation runs from the CLI: `midas-ff-zip -resultFolder
… -paramFN … -LayerNr …`. HDF5, GE, and CBF inputs are auto-detected
from the data file extension. See the package
[README](../README.md).
